# PySpark 10: batch pipelines

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-10-batch-pipelines.ipynb)

A nightly job publishes the order tables finance reports from. It must replace only the days it rebuilt, apply
late changes exactly once, keep the history of changing customer attributes, stop bad data before it reaches a
dashboard, and give the same result when the scheduler retries it.

This lesson builds those guarantees one at a time: the medallion layout, partition-scoped overwrites, incremental
loads with a watermark, deduplicating a change feed, version-aware merges into a Delta table, slowly changing
dimensions, quality gates and quarantines, reconciliation between layers, and backfills.

Each section explains how something works, where it matters and what to watch out for, then runs small examples. Read each output, then the **What to notice** note under it. Run the cells in order.

**Prerequisites:** [PySpark 4: reading and writing data](pyspark-04-reading-and-writing-data.ipynb), [PySpark 5: Spark SQL and the catalog](pyspark-05-spark-sql-and-the-catalog.ipynb) and [PySpark 7: aggregations and windows](pyspark-07-aggregations-and-windows.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0 deltalake pyarrow

## Setup and sample data

The first cell starts a local Spark session. The second creates `customers` (one row per customer; customer 4 has no region, and 7 and 8 have never ordered), `orders` (one row per order; order 12 lands just after midnight UTC) and `order_items` (one row per order line; each order's lines add up to its amount): a slice of the orders platform that every
module shares, introduced in [PySpark 1](pyspark-01-how-spark-runs.ipynb#setup-and-sample-data).

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import functions as F
from pyspark.sql import Window

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

order_items = spark.createDataFrame([
    (1, "SKU-4001", 2, D("12.50")), (1, "SKU-4007", 1, D("95.00")),
    (2, "SKU-4002", 1, D("34.50")), (2, "SKU-4009", 2, D("25.00")),
    (3, "SKU-4002", 1, D("30.00")),
    (4, "SKU-4003", 1, D("149.00")), (4, "SKU-4004", 2, D("55.50")),
    (5, "SKU-4005", 1, D("55.25")),
    (6, "SKU-4003", 1, D("149.00")),
    (7, "SKU-4006", 1, D("189.00")), (7, "SKU-4008", 1, D("121.00")),
    (8, "SKU-4010", 1, D("89.90")),
    (9, "SKU-4011", 20, D("59.00")),  # a bulk reorder
    (10, "SKU-4012", 2, D("37.00")),
    (11, "SKU-4013", 1, D("210.00")),
    (12, "SKU-4014", 1, D("41.40")), (12, "SKU-4005", 1, D("55.00")),
], "order_id int, sku string, qty int, unit_price decimal(8,2)")

In [ ]:
def table_totals(path, by="order_date"):
    """Rows and revenue per partition value, for before/after comparisons."""
    df = spark.read.parquet(path)
    return {str(r[by]): (r["rows"], r["revenue"]) for r in
            df.groupBy(by).agg(F.count("*").alias("rows"), F.sum("amount").alias("revenue")).collect()}

## Design the pipeline in layers

The **medallion** layout splits a pipeline into layers, each rebuildable from the one before:

| layer | holds | typical rules |
| --- | --- | --- |
| **bronze** | raw data as delivered, typed with a declared schema, plus lineage columns | never edited; bad rows quarantined, not dropped |
| **silver** | cleaned, deduplicated, one row per business key, conformed types and names | merges and corrections land here |
| **gold** | tables shaped for consumers: aggregates, star schemas, features | rebuilt from silver |

Two properties make such a pipeline safe to operate. It is **idempotent**: running the same input twice leaves the
same output. It is **deterministic**: the same input always gives the same output, with no dependence on the clock
or on row order.

**Where it matters**
- **Bug fixes become reruns:** fix the code, rebuild silver and gold from bronze.
- **Audits:** bronze keeps exactly what each source delivered and when.

**Watch out for**
- `current_timestamp()` in data and `dropDuplicates` without an order break determinism.
- Appending in a job the scheduler can retry breaks idempotency.

### Example

The rest of this lesson builds each property separately. The [end-to-end capstone](pyspark-concepts-end-to-end.ipynb)
assembles them into one pipeline from raw files to gold tables.

## Overwrite only the partitions you rebuild

On a partitioned table, `mode("overwrite")` replaces the **whole** table by default. With
`spark.sql.sources.partitionOverwriteMode=dynamic`, only the partitions present in the DataFrame are replaced and
every other partition keeps its files. Delta Lake and Iceberg offer the same with `replaceWhere` and
`overwritePartitions()`.

**Where it matters**
- **Reprocessing one day** after a bug fix without touching the other 364.
- **Daily snapshot loads:** each run replaces its own business date.

**Watch out for**
- A date whose correct result is now empty has no rows, so dynamic overwrite never replaces it; delete it explicitly.
- The rewritten partition must keep the table's column types. Plain Parquet does not check, so one partition written as `double` breaks every later read.

### Example

A pricing error on 12 January is fixed. Rewrite only that day and compare every day's totals before and after.

In [ ]:
silver_path = os.path.join(WORK_DIR, "silver_orders")
with_date = orders.withColumn("order_date", F.to_date("order_ts"))
with_date.write.mode("overwrite").partitionBy("order_date").parquet(silver_path)
before = table_totals(silver_path)

amount_type = with_date.schema["amount"].dataType  # keep the column's type identical
corrected = (
    with_date.where("order_date = '2026-01-12'")
    .withColumn("amount", (F.col("amount") * F.lit(D("0.9"))).cast(amount_type))
)
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
try:
    corrected.write.mode("overwrite").partitionBy("order_date").parquet(silver_path)
finally:
    spark.conf.set("spark.sql.sources.partitionOverwriteMode", "static")

after = table_totals(silver_path)
for day in sorted(before):
    print(day, before[day], "->", after[day], "" if before[day] == after[day] else "<- rewritten")

**What to notice:** Only 12 January changed: its revenue fell by 10 percent and every other day kept the same rows and totals. With
the default static mode, the same write would have deleted all the other days. The correction was cast back to
the column's decimal type; multiplying by a Python float would have written that one partition as `double`.

## Load only new data with a watermark

An incremental job processes only what arrived since the last successful run. Keep the **high-water mark** (the
latest source timestamp or sequence number loaded) in a small operations table, read only newer rows, process
them, and advance the mark **after** the target write commits. Re-read a short lookback window to catch rows that
arrive late.

**Where it matters**
- **Hourly or daily loads** from a landing zone or an operational database.
- **Large sources** where a full reload is too slow or too expensive.

**Watch out for**
- Deriving the mark from `max(...)` of the target skips late rows forever; keep it in an operations table.
- Advancing the mark before the write commits loses a batch if the write fails.

### Example

Run an incremental load twice: the first run loads everything after the mark, the second finds nothing new.

In [ ]:
ops_path = os.path.join(WORK_DIR, "ops_watermarks")
spark.createDataFrame([("orders", datetime(2026, 1, 10))], "pipeline string, loaded_until timestamp") \
    .write.mode("overwrite").parquet(ops_path)
target_path = os.path.join(WORK_DIR, "orders_incremental")
with_date.where("order_ts <= '2026-01-10'").write.mode("overwrite").parquet(target_path)


def run_incremental():
    mark = spark.read.parquet(ops_path).where("pipeline = 'orders'").first()["loaded_until"]
    new_rows = with_date.where(F.col("order_ts") > F.lit(mark))
    count = new_rows.count()
    if count:
        new_rows.write.mode("append").parquet(target_path)
        new_mark = new_rows.agg(F.max("order_ts")).first()[0]
        spark.createDataFrame([("orders", new_mark)], "pipeline string, loaded_until timestamp") \
            .write.mode("overwrite").parquet(ops_path)  # advance only after the write succeeded
    return mark, count

for run in (1, 2):
    mark, count = run_incremental()
    print(f"run {run}: mark was {mark}, loaded {count} rows, target now {spark.read.parquet(target_path).count()} rows")

**What to notice:** The first run read the mark (10 January), loaded the 5 newer orders and moved the mark to the latest order time.
The second run found nothing newer, so it wrote nothing and the target stayed at 12 rows. The append is safe here
only because the mark moves after the write; the next section handles a change feed where the same key can
arrive again.

## Apply a change feed exactly once

A change feed (CDC) can deliver several versions of a key in one batch, and a retry can deliver the whole batch
again. To apply it safely:

1. keep one change per key with `row_number` over the source sequence ([lesson 7](pyspark-07-aggregations-and-windows.ipynb));
2. **upsert**: replace target rows whose key appears in the batch and insert new keys.

Delta Lake and Iceberg do step 2 with `MERGE INTO`. On plain Parquet, the same result is "target rows whose key is
not in the batch, plus the batch", written to a new location.

**Where it matters**
- **Silver tables** fed by Debezium or database change streams.
- **Corrections** sent by a vendor for orders already loaded.

**Watch out for**
- Two changes for one key in a batch make `MERGE` fail or pick arbitrarily; deduplicate first.
- Without a source version check, a replayed old batch overwrites newer data (next section).

### Example

Order 8 was refunded and the change arrived twice; order 13 is new. Deduplicate, upsert, and apply the same batch again.

In [ ]:
change_schema = "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2), source_lsn int, ingested_at timestamp"
batch = spark.createDataFrame(
    [(8, 4, "refunded", datetime(2026, 1, 10, 19, 30), D("89.90"), 7, datetime(2026, 1, 15, 8, 0)),
     (8, 4, "refunded", datetime(2026, 1, 10, 19, 30), D("89.90"), 7, datetime(2026, 1, 15, 8, 5)),
     (13, 2, "placed", datetime(2026, 1, 15, 9, 0), D("42.00"), 8, datetime(2026, 1, 15, 9, 1))],
    change_schema,
)
newest = Window.partitionBy("order_id").orderBy(F.desc("source_lsn"), F.desc("ingested_at"))
staged = batch.withColumn("rn", F.row_number().over(newest)).where("rn = 1").drop("rn", "source_lsn", "ingested_at")


def upsert(target, changes, key="order_id"):
    return target.join(changes.select(key), key, "left_anti").unionByName(changes)

silver = orders
once = upsert(silver, staged)
twice = upsert(once, staged)
print("staged changes:", staged.count(), "| rows after one run:", once.count(), "| after two runs:", twice.count())
print("differences between one and two runs:", once.exceptAll(twice).count() + twice.exceptAll(once).count())
once.where("order_id IN (8, 13)").orderBy("order_id").show()

**What to notice:** The duplicate refund was staged once, so the batch had two changes. Silver went from 12 to 13 orders: order 8 was
replaced and order 13 added. Applying the same batch a second time changed nothing, which is exactly what makes a
retry harmless.

## Merge into a Delta table, version by version

With a table format, the upsert is one atomic `MERGE`. Add an update condition so only a **newer** source version
can replace a row: then a retry is a no-op, and an old delivery replayed late cannot overwrite newer data. In
PySpark with the `delta-spark` package:

```python
from delta.tables import DeltaTable

(
    DeltaTable.forName(spark, "main.silver.orders").alias("t")
    .merge(staged.alias("s"), "t.order_id = s.order_id")
    .whenMatchedUpdateAll(condition="s.source_version > t.source_version")
    .whenNotMatchedInsertAll()
    .execute()
)
```

`delta-spark` needs Delta's JARs on the Spark classpath, which managed platforms provide. To run without
downloading them, the example uses the [`deltalake`](https://delta-io.github.io/delta-rs/) Python package, which
writes the same Delta format with the same merge semantics.

**Where it matters**
- **Retried batches** after a failure between the commit and the scheduler recording success.
- **Out-of-order deliveries:** Monday's file resent on Wednesday, after Tuesday's corrections.

**Watch out for**
- Each source batch needs at most one row per key; deduplicate before merging.
- Old versions stay readable only until `VACUUM` removes their files.

### Example

Order 1 starts at source version 1. A correction at version 2 raises its amount from 10 to 15. Apply the
correction twice, then replay the original version-1 delivery, and read the table's history and an old version.

In [ ]:
import pyarrow as pa
from deltalake import DeltaTable, write_deltalake


def merge_orders(table, changes):
    (
        table.merge(changes, predicate="t.id = s.id", source_alias="s", target_alias="t")
        .when_matched_update_all(predicate="s.source_version > t.source_version")
        .when_not_matched_insert_all()
        .execute()
    )

delta_path = os.path.join(WORK_DIR, "delta_orders")
initial = pa.table({"id": [1, 2], "amount": [10, 20], "source_version": [1, 1]})
write_deltalake(delta_path, initial)
table = DeltaTable(delta_path)
correction = pa.table({"id": [1, 3], "amount": [15, 30], "source_version": [2, 1]})

merge_orders(table, correction)
merge_orders(table, correction)  # a retry
merge_orders(table, initial.slice(0, 1))  # the stale version-1 delivery, replayed late

print("current:  ", sorted(table.to_pyarrow_table().to_pylist(), key=lambda r: r["id"]))
print("version 0:", DeltaTable(delta_path, version=0).to_pyarrow_table().to_pylist())
print("history:  ", [entry["operation"] for entry in reversed(table.history())])

**What to notice:** Order 1 ends at amount 15, version 2, and order 3 was inserted. The retry matched but changed nothing, and the
stale replay failed the `s.source_version > t.source_version` condition, so the history shows a single `MERGE`:
no-op merges commit no new version. Version 0 is still readable through time travel.

## Keep history with a slowly changing dimension

A **slowly changing dimension type 2** (SCD2) keeps every version of an attribute, with `valid_from`, `valid_to`
(null while current) and `is_current`. When a change arrives, the current row is closed at the change date and a
new current row opened; unchanged keys are left alone and new keys inserted. Facts then join to the version valid
at the event time ([lesson 6](pyspark-06-joins.ipynb)). On Delta this is a single `MERGE` over a staged set of rows to close and
rows to insert; here the steps are spelled out with DataFrames.

**Where it matters**
- **Customer region or segment history,** so last year's revenue stays with last year's region.
- **Audits:** what did we believe about this customer on 1 March?

**Watch out for**
- Compare attributes null-safely; a change from null to a value is a change.
- Exactly one current row per key is the invariant; check it after every load.
- Rerunning the same batch must not open new versions: compare with the current value, not just the key.

### Example

Customer 1 moves from North to West, customer 2 sends an unchanged record, and customer 3 is new. Apply the changes twice.

In [ ]:
dim_schema = "customer_id int, region string, valid_from date, valid_to date, is_current boolean"
dim = spark.createDataFrame(
    [(1, "North", date(2024, 1, 3), None, True), (2, "North", date(2024, 2, 11), None, True)], dim_schema
)
changes = spark.createDataFrame(
    [(1, "West", date(2026, 1, 15)), (2, "North", date(2026, 1, 15)), (3, "South", date(2026, 1, 15))],
    "customer_id int, region string, changed_on date",
)


def apply_scd2(dim, changes):
    current = dim.where("is_current").select("customer_id", F.col("region").alias("current_region"))
    changed = changes.join(current, "customer_id", "left").where(~F.col("region").eqNullSafe(F.col("current_region")))
    closing = changed.select("customer_id", "changed_on")
    closed = (
        dim.join(closing, "customer_id", "left")
        .withColumn("closes", F.col("is_current") & F.col("changed_on").isNotNull())
        .withColumn("valid_to", F.when(F.col("closes"), F.col("changed_on")).otherwise(F.col("valid_to")))
        .withColumn("is_current", F.col("is_current") & ~F.col("closes"))
        .drop("changed_on", "closes")
    )
    opened = changed.select("customer_id", "region", F.col("changed_on").alias("valid_from"),
                            F.lit(None).cast("date").alias("valid_to"), F.lit(True).alias("is_current"))
    return closed.unionByName(opened)

dim_v2 = apply_scd2(dim, changes)
dim_v2.orderBy("customer_id", "valid_from").show()
dim_v2.groupBy("customer_id").agg(F.sum(F.col("is_current").cast("int")).alias("current_rows")).orderBy("customer_id").show()
dim_v3 = apply_scd2(dim_v2, changes)
print("rows changed by a rerun:", dim_v3.exceptAll(dim_v2).count() + dim_v2.exceptAll(dim_v3).count())

**What to notice:** Customer 1 now has two rows: North, closed on 15 January, and West, current from that day. Customer 2's unchanged
record opened nothing, and customer 3 was inserted. Every customer has exactly one current row, and the rerun
changed nothing because each change is compared with the current value.

## Check quality, quarantine bad rows and gate publishing

Evaluate every rule once into a reason column, then split the data: valid rows continue, rejected rows go to a
quarantine table with the reason and the batch id. Some rules are **hard** (a missing key: never publish) and some
are **soft** (an unknown customer: publish but warn). A **quality gate** before publishing raises an error when a
hard rule fails, so a broken batch stops the pipeline instead of reaching a dashboard.

**Where it matters**
- **Before every gold publish:** keys unique and not null, amounts in range, statuses known.
- **Feedback to producers:** quarantine counts by reason are the report you send back.

**Watch out for**
- Dropping bad rows silently hides problems; quarantine them and count them.
- Both outputs read the checked data; cache it, or the rules run twice.
- Gates should fail loudly with the counts, not with a bare `assert` that is easy to disable.

### Example

Check five incoming orders against three rules, quarantine the failures, and run a gate on the valid rows.

In [ ]:
incoming = spark.createDataFrame(
    [(1, 10, D("50.00"), "paid"), (2, None, D("20.00"), "paid"), (None, 11, D("5.00"), "placed"),
     (4, 12, D("-3.00"), "paid"), (5, 13, D("8.00"), "teleported")],
    "order_id int, customer_id int, amount decimal(12,2), status string",
)
checked = incoming.withColumn(
    "reason",
    F.when(F.col("order_id").isNull(), "missing order_id")
    .when(F.col("customer_id").isNull(), "missing customer_id")
    .when(F.col("amount") < 0, "negative amount"),
).cache()
valid = checked.where("reason IS NULL").drop("reason")
quarantine = checked.where("reason IS NOT NULL").withColumn("batch_id", F.lit("2026-01-15"))
quarantine.groupBy("reason").count().orderBy("reason").show()


def quality_gate(df):
    report = {
        "duplicate_keys": df.groupBy("order_id").count().where("count > 1").count(),
        "unknown_status": df.where(~F.col("status").isin("placed", "paid", "shipped", "returned")).count(),
    }
    print("quality report:", report)
    if any(report.values()):
        raise ValueError(f"quality gate failed, not publishing: {report}")

try:
    quality_gate(valid)
except ValueError as error:
    print(error)
print("input:", incoming.count(), "| valid:", valid.count(), "| quarantined:", quarantine.count())
checked = checked.unpersist()

**What to notice:** Three rows were quarantined, one reason each. The two remaining rows include order 5 with the status
`teleported`, which no row-level rule caught; the gate's table-level check did, and refused to publish. Input
rows equal valid plus quarantined rows, so nothing disappeared.

## Reconcile between layers

**Reconciliation** proves that every input record is accounted for. At each boundary, compare counts **and**
totals at the same grain and filter: landed rows = accepted + rejected; silver rows = accepted - duplicates;
gold revenue = the sum of eligible silver amounts. A count alone can hide offsetting errors (one row lost, one
duplicated), so check amounts too.

**Where it matters**
- **Finance tables:** gold revenue must match the source system to the cent.
- **After backfills and migrations,** before switching readers to the new data.

**Watch out for**
- Compare like with like: the same business dates and the same status filter on both sides.
- Track business dates separately from delivery dates, so late rows are reconciled against the day they belong to.

### Example

Reconcile order lines with orders, and a gold daily-revenue table with silver orders.

In [ ]:
lines_per_order = order_items.groupBy("order_id").agg(F.sum(F.col("qty") * F.col("unit_price")).alias("lines_total"))
mismatched = lines_per_order.join(orders, "order_id", "full").where(~F.col("lines_total").eqNullSafe(F.col("amount")))
print("orders whose lines do not add up:", mismatched.count())

eligible = orders.where(F.col("status").isin("paid", "shipped"))
gold = eligible.groupBy(F.to_date("order_ts").alias("order_date")).agg(F.sum("amount").alias("revenue"))
silver_total = eligible.agg(F.sum("amount")).first()[0]
gold_total = gold.agg(F.sum("revenue")).first()[0]
print("silver eligible revenue:", silver_total, "| gold revenue:", gold_total, "| match:", silver_total == gold_total)

**What to notice:** Every order's lines add up to its amount, and gold revenue equals eligible silver revenue exactly, because both
sides use decimals and the same status filter. The `full` join with `eqNullSafe` would also catch orders with no
lines and lines with no order.

## Backfill a slice without duplicating data

A **backfill** rebuilds a historical slice from retained input, for example after a bug fix or a late delivery.
Name the affected business dates first, then decide per delivery: a full replacement overwrites the slice
(dynamic partition overwrite); a set of changes is merged by key and version. Run the backfill with the same code
as the daily job and a date parameter.

**Where it matters**
- **A vendor sends Monday's missing orders on Wednesday:** rebuild Monday, leave Tuesday alone.
- **Reprocessing a month** after fixing a transformation.

**Watch out for**
- Dates whose correct result is now empty are not replaced by dynamic overwrite; delete them explicitly.
- Other jobs writing the same slice at the same time.

### Example

Monday has three orders and Tuesday two. A late delivery adds one Monday order. Rebuild Monday from all its input, twice.

In [ ]:
backfill_path = os.path.join(WORK_DIR, "orders_by_business_date")
schema = "order_id int, order_date date, amount decimal(12,2)"
monday, tuesday = date(2026, 1, 12), date(2026, 1, 13)
delivered = spark.createDataFrame(
    [(1, monday, D("10.00")), (2, monday, D("25.00")), (3, monday, D("5.00")), (4, tuesday, D("12.00")), (5, tuesday, D("8.00"))], schema
)
late = spark.createDataFrame([(6, monday, D("40.00"))], schema)
delivered.write.mode("overwrite").partitionBy("order_date").parquet(backfill_path)
tuesday_folder = os.path.join(backfill_path, "order_date=2026-01-13")
tuesday_files = sorted(os.listdir(tuesday_folder))


def backfill(dates):
    rebuilt = delivered.unionByName(late).where(F.col("order_date").isin(dates))
    spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
    try:
        rebuilt.write.mode("overwrite").partitionBy("order_date").parquet(backfill_path)
    finally:
        spark.conf.set("spark.sql.sources.partitionOverwriteMode", "static")

print("before:      ", table_totals(backfill_path))
backfill([monday])
print("after:       ", table_totals(backfill_path))
backfill([monday])  # the scheduler retries
print("after retry: ", table_totals(backfill_path))
print("Tuesday's files untouched:", sorted(os.listdir(tuesday_folder)) == tuesday_files)

**What to notice:** Monday went from 3 orders (40.00) to 4 (80.00) once the late order was included, and the retry left the same
totals. Tuesday kept the same files: only the `order_date=2026-01-12` folder was replaced.

## Check yourself

Answer each question in your own words before reading the answer.

**Q1.** What makes a pipeline idempotent, and why does it matter?

**A.** Running it twice on the same input leaves the same output, so a scheduler retry after a partial failure is harmless. Writes are scoped to what the batch owns (overwrite a slice, or merge by key) instead of appending.

**Q2.** Dynamic partition overwrite is on and a corrected day now has no rows. What happens?

**A.** Nothing: the day has no rows in the output, so its old partition stays. Delete it explicitly.

**Q3.** Where should an incremental job keep its watermark, and when should it advance it?

**A.** In an operations table, not derived from the target; advance it only after the target write commits.

**Q4.** How does a merge stay correct when an old delivery is replayed after a newer one?

**A.** The update condition only lets a newer source version win, so the stale rows match but do not update.

**Q5.** What invariant must an SCD2 dimension satisfy?

**A.** Exactly one current row per key, with non-overlapping validity intervals; rerunning a batch opens no new versions.

**Q6.** Why reconcile totals as well as counts?

**A.** Counts can balance while amounts do not: one row lost and one duplicated leave the count unchanged.

## Key takeaways

- Layer pipelines as bronze, silver and gold, each rebuildable from the one before.
- Scope every write to what the batch owns: dynamic partition overwrite or a merge by key.
- Deduplicate change feeds by source sequence, and guard merges with a version condition.
- SCD2: close the current row, open a new one, keep one current row per key.
- Quarantine bad rows with a reason, gate publishing on hard rules, and reconcile counts and totals.
- Backfills reuse the daily job with a date parameter and are proven by a rerun.

**Next:** Practise this module in [PySpark drills 10: batch pipelines](pyspark-drills-10-batch-pipelines.ipynb), then continue with [PySpark 11: structured streaming](pyspark-11-structured-streaming.ipynb).